# Migration Phase 2: Anatomical Tabular Derivatives (`TabularDerivativesLoader.load_anatomical`)

`src/neuroalign/data/loaders/tabular_derivatives.py` replaces `AnatomicalLoader`
(CAT12 + MATLAB TIV, on-the-fly parcellation). Anatomical data is now
**pre-parcellated** as flat CSVs under `TABULAR_DERIVATIVES_ROOT`.

For each `(uid, session_id)`:
- resolves `sub-<uid>/ses-<session_id>{.cross,}/anat/` per `SESSION_VARIANT` (default `cross`)
- loads `atlas-Schaefer2018N400n7` (cortex) + `atlas-Tian2020S2` (subcortex)
- concatenates into one long-format table labeled `atlas=Schaefer2018N400n7Tian2020S2`
- drops the `Background+FreeSurfer_Defined_Medial_Wall` rows (not real regions, absent from dwi atlases)

**TIV is now per-row (`tiv_mm3`)** - no MATLAB/CAT12 subprocess step needed.

In [1]:
from pathlib import Path
from dotenv import load_dotenv
import os

load_dotenv(Path.cwd().parent / ".env")

from neuroalign.data.loaders import BehavioralLoader, TabularDerivativesLoader

beh = BehavioralLoader(os.environ["BRAINLINK_DB_PATH"])
loader = TabularDerivativesLoader(
    os.environ["TABULAR_DERIVATIVES_ROOT"],
    atlas_name=os.environ["ATLAS_NAME"],
    anat_atlases=tuple(os.environ["ANAT_ATLASES"].split(",")),
    session_variant=os.environ["SESSION_VARIANT"],
)
print(f"atlas_name={loader.atlas_name!r} anat_atlases={loader.anat_atlases!r} session_variant={loader.session_variant!r}")

atlas_name='Schaefer2018N400n7Tian2020S2' anat_atlases=('Schaefer2018N400n7', 'Tian2020S2') session_variant='cross'


## 1. Load anatomical data for a sample of sessions

Loading all ~4,900 sessions takes a few minutes (one CSV read per atlas per
session); a 200-session sample is enough to validate the shape and content
here. The full set is loaded once in the Phase 5 pipeline run.

In [2]:
sessions = beh.get_sessions()
sample = sessions.head(200)

anat = loader.load_anatomical(sample)
print(f"{len(anat)} rows, {anat['uid'].nunique()} / {sample['uid'].nunique()} sample uids covered")
print(anat.columns.tolist())

25488 rows, 58 / 197 sample uids covered
['uid', 'session_id', 'index', 'label', 'hemisphere', 'num_vertices', 'surface_area_mm2', 'gray_matter_volume_mm3', 'thickness_mean_mm', 'thickness_std_mm', 'mean_curvature', 'gaussian_curvature', 'folding_index', 'curvature_index', 'white_surf_area_mm2', 'brain_seg_vol_mm3', 'brain_seg_no_vent_mm3', 'cortex_vol_mm3', 'supratentorial_vol_mm3', 'tiv_mm3', 'structure', 'atlas', 'num_voxels', 'volume_mm3', 'intensity_mean', 'intensity_std', 'intensity_min', 'intensity_max', 'intensity_range', 'intensity_snr', 'subcort_gray_mm3']


## 2. Cortex vs. subcortex

Cortex (`Schaefer2018N400n7`) and subcortex (`Tian2020S2`) rows have **different
metric columns** - surface-based (`thickness_mean_mm`, `surface_area_mm2`, ...) vs.
volume-based (`volume_mm3`, `intensity_mean`, ...). Both share `tiv_mm3`.

In [3]:
print(anat["structure"].value_counts())
print()
print("cortex columns:", [c for c in anat.columns if anat[anat["structure"] == "cortex"][c].notna().any() and anat[anat["structure"] == "subcortex"][c].isna().all()])
print()
print("subcortex columns:", [c for c in anat.columns if anat[anat["structure"] == "subcortex"][c].notna().any() and anat[anat["structure"] == "cortex"][c].isna().all()])
print()
print("shared columns:", [c for c in anat.columns if anat[anat["structure"] == "cortex"][c].notna().any() and anat[anat["structure"] == "subcortex"][c].notna().any()])

structure
cortex       23600
subcortex     1888
Name: count, dtype: int64

cortex columns: ['num_vertices', 'surface_area_mm2', 'gray_matter_volume_mm3', 'thickness_mean_mm', 'thickness_std_mm', 'mean_curvature', 'gaussian_curvature', 'folding_index', 'curvature_index', 'white_surf_area_mm2', 'brain_seg_vol_mm3', 'brain_seg_no_vent_mm3', 'cortex_vol_mm3', 'supratentorial_vol_mm3']



subcortex columns: ['num_voxels', 'volume_mm3', 'intensity_mean', 'intensity_std', 'intensity_min', 'intensity_max', 'intensity_range', 'intensity_snr', 'subcort_gray_mm3']

shared columns: ['uid', 'session_id', 'index', 'label', 'hemisphere', 'tiv_mm3', 'structure', 'atlas']


## 3. Region counts and TIV

400 cortical regions (`7Networks_*`, 200 per hemisphere) + 32 subcortical
regions (`Tian2020S2`, e.g. `aHIP-rh`) = 432 regions per session.

In [4]:
one_session = anat[(anat["uid"] == anat["uid"].iloc[0]) & (anat["session_id"] == anat["session_id"].iloc[0])]
print(f"{len(one_session)} regions for sub-{one_session['uid'].iloc[0]} ses-{one_session['session_id'].iloc[0]}")
print("sample cortex labels:", one_session[one_session["structure"] == "cortex"]["label"].head(3).tolist())
print("sample subcortex labels:", one_session[one_session["structure"] == "subcortex"]["label"].head(3).tolist())
print("tiv_mm3 (constant per session):", one_session["tiv_mm3"].unique())

432 regions for sub-S004192 ses-202012061756
sample cortex labels: ['7Networks_LH_DorsAttn_Post_1', '7Networks_LH_DorsAttn_Post_2', '7Networks_RH_DorsAttn_Post_2']
sample subcortex labels: ['aHIP-rh', 'pHIP-rh', 'lAMY-rh']
tiv_mm3 (constant per session): [1598223.027001]


## 4. Note: cortex region-name prefix vs. diffusion atlas

`Schaefer2018N400n7` cortex labels carry a `7Networks_` prefix (e.g.
`7Networks_LH_DorsAttn_Post_1`), while the diffusion `Schaefer2018N400n7Tian2020S2`
atlas (Phase 3) uses the same names **without** the prefix (e.g. `LH_DorsAttn_Post_1`).
Subcortical (`Tian2020S2`) labels match exactly between anat and dwi (e.g. `aHIP-rh`).

This only matters for `regional-stacker`'s region alignment across modalities
(Phase 7) - flagged here, addressed when building the multimodal wide tables.